# Notebook 3 — Generation Stage

For each query × method, sends the retrieved documents to the LLM and extracts citation order.

**Input:** `data/retail/retrieval_results.csv`

Columns: `query_id | query | method | target_doc_id | target_doc_rank | rank | doc_text | is_target`

**Output:** `data/retail/generation_results.parquet`

Columns: `query_id | query | method | target_doc_id | citation_order | boost_product_index | llm_response`

## Instructions
1. Run **Setup**
2. Run **Parameters**
3. Run **Generation**
4. Run **Inspect Results**
5. Run **Summary**

## Setup

In [ ]:
import json
import os
import re
import sys
import time
from datetime import datetime

import pandas as pd
import numpy as np

notebook_dir = os.getcwd()
project_root = os.path.abspath(os.path.join(notebook_dir, ".."))
sys.path.insert(0, os.path.join(project_root, "src"))

from llms import OpenAIHelper

config_path = os.path.join(project_root, "config.json")
with open(config_path, "r") as f:
    config = json.load(f)
os.environ["OPENAI_API_KEY"] = config["OPENAI_API_KEY"]

data_dir = os.path.join(project_root, "data", "retail")

print("Setup complete.")
print(f"Project root: {project_root}")

## Helper Functions

In [ ]:
DEVELOPER_PROMPT = """Your are a product recommender system for an online marketplace. Write an accurate and concise answer for the given user question, using _only_ the provided summarized web search results. The answer should be correct, high-quality, and written by an expert using an unbiased and journalistic tone. The answer should be informative, interesting, and engaging. The answer's logic and reasoning should be rigorous and defensible. Every sentence in the answer should be _immediately followed_ by an in-line citation to the search result(s). The cited search result(s) should fully support _all_ the information in the sentence. Search results need to be cited using [index]. When citing several search results, use [1][2][3] format rather than [1, 2, 3]. You can use multiple search results to respond comprehensively while avoiding irrelevant search results. Search results are not sorted."""
DOC_TYPE = "Product Description"


def build_user_prompt(query, docs):
    """
    Builds the user prompt for the LLM.
    docs: list of doc_text strings, already sorted by rank and re-numbered 1..N
    """
    sources = ""
    for i, doc_text in enumerate(docs, start=1):
        sources += f"[{i}] {DOC_TYPE}: {doc_text}\n\n"
    return f"Question: {query}\n\nSearch Results:\n{sources}"


def extract_citation_order(response_text):
    """
    Extracts citation indices in order of first appearance from LLM response.
    0-based like Puerto: [1] in text -> index 0, [2] -> index 1, etc.
    e.g. "...product [2] is great [1][2]..." -> [1, 0]
    """
    citations = re.findall(r'\[(\d+)\]', response_text)
    seen = []
    for c in citations:
        idx = int(c) - 1  # 0-based like Puerto
        if idx not in seen:
            seen.append(idx)
    return seen


def build_context_for_method(group, method):
    """
    Given all rows for one query, builds the document list for one method.
    
    - Keeps all competitor docs (is_target=0)
    - Adds exactly one target doc version (is_target=1, method=method)
    - Sorts by original rank
    - Re-numbers positions 1..N consecutively
    
    Returns:
        docs: list of doc_text in order
        target_new_position: 1-based position of target doc after re-numbering
    """
    competitors = group[group["is_target"] == 0].copy()
    target_rows = group[(group["is_target"] == 1) & (group["method"] == method)].copy()

    if len(target_rows) == 0:
        return None, None

    target_row = target_rows.iloc[0]
    combined = pd.concat([competitors, target_rows.iloc[[0]]], ignore_index=True)
    combined = combined.sort_values("rank").reset_index(drop=True)

    # Re-number positions 1..N
    docs = combined["doc_text"].tolist()

    # Find new position of target doc (0-based like Puerto)
    target_new_position = None
    for i, row in combined.iterrows():
        if row["is_target"] == 1:
            target_new_position = i  # 0-based
            break

    return docs, target_new_position


print("Helper functions defined.")

## Parameters

In [ ]:
# LLM
LLM_NAME = "gpt-4o-mini-2024-07-18"

# Input
retrieval_csv_path = os.path.join(data_dir, "retrieval_results.csv")

# Output
output_path = os.path.join(data_dir, "generation_results.parquet")

# Load retrieval results
df = pd.read_csv(retrieval_csv_path)

# Get all methods
all_methods = df[df["is_target"] == 1]["method"].unique().tolist()
all_query_ids = df["query_id"].unique().tolist()

# Calculate total LLM calls needed
total_calls = len(all_query_ids) * len(all_methods)

# Check already done if output exists
if os.path.exists(output_path):
    df_done = pd.read_parquet(output_path)
    already_done = len(df_done)
else:
    already_done = 0

print(f"LLM:             {LLM_NAME}")
print(f"Total queries:   {len(all_query_ids)}")
print(f"Methods:         {len(all_methods)} — {all_methods}")
print(f"Total LLM calls: {total_calls}")
print(f"Already done:    {already_done}")
print(f"Remaining:       {total_calls - already_done}")
print(f"Estimated cost:  ~${(total_calls - already_done) * 0.0003:.2f}")

## Generation

For each query × method:
1. Build document context (competitors + one target version)
2. Re-number positions 1..N
3. LLM generates response with inline citations
4. Extract citation order
5. Save incrementally

Safe to interrupt and resume.

In [ ]:
llm = OpenAIHelper(LLM_NAME)

# Load existing results if any
if os.path.exists(output_path):
    results_df = pd.read_parquet(output_path)
    results = results_df.to_dict("records")
    done_keys = set(zip(results_df["query_id"], results_df["method"]))
else:
    results = []
    done_keys = set()

start_time = datetime.now()
print(f"Started at: {start_time.strftime('%H:%M:%S')}")
print()

for query_id in all_query_ids:
    group = df[df["query_id"] == query_id]
    query_text = group["query"].iloc[0]
    target_doc_id = group[group["is_target"] == 1]["target_doc_id"].iloc[0]

    for method in all_methods:
        # Skip if already done
        if (query_id, method) in done_keys:
            continue

        # Build context
        docs, target_new_position = build_context_for_method(group, method)

        if docs is None:
            print(f"  [{query_id}] {method}: no target doc found — skipping")
            continue

        # Build prompt
        user_prompt = build_user_prompt(query_text, docs)
        messages = [
            {"role": "system", "content": DEVELOPER_PROMPT},
            {"role": "user", "content": user_prompt},
        ]

        try:
            response, _ = llm.generate(messages)
            response_text = response.content

            citation_order = extract_citation_order(response_text)

            results.append({
                "query_id":            query_id,
                "query":               query_text,
                "method":              method,
                "target_doc_id":       target_doc_id,
                "target_new_position": target_new_position,
                "num_docs":            len(docs),
                "citation_order":      citation_order,
                "boost_product_index": target_new_position,
                "llm_response":        response_text,
            })
            done_keys.add((query_id, method))

            # Save incrementally
            pd.DataFrame(results).to_parquet(output_path, index=False)
            print(f"  [{query_id}] {method}: cited at positions {citation_order[:5]} | target at [{target_new_position}]")

        except Exception as e:
            print(f"  [{query_id}] {method}: ERROR — {e}")
            time.sleep(10)
            try:
                response, _ = llm.generate(messages)
                response_text = response.content
                citation_order = extract_citation_order(response_text)
                results.append({
                    "query_id":            query_id,
                    "query":               query_text,
                    "method":              method,
                    "target_doc_id":       target_doc_id,
                    "target_new_position": target_new_position,
                    "num_docs":            len(docs),
                    "citation_order":      citation_order,
                    "boost_product_index": target_new_position,
                    "llm_response":        response_text,
                })
                done_keys.add((query_id, method))
                pd.DataFrame(results).to_parquet(output_path, index=False)
                print(f"  [{query_id}] {method}: done (retry OK)")
            except Exception as e2:
                print(f"  [{query_id}] {method}: FAILED — {e2}")

end_time = datetime.now()
elapsed = end_time - start_time
print(f"{'='*60}")
print(f"GENERATION COMPLETE")
print(f"Started:    {start_time.strftime('%H:%M:%S')}")
print(f"Finished:   {end_time.strftime('%H:%M:%S')}")
print(f"Total time: {str(elapsed).split('.')[0]}")
print(f"Results:    {len(results)} rows saved to {output_path}")

## Inspect Results

In [ ]:
df_results = pd.read_parquet(output_path)

INSPECT_QUERY_ID = df_results["query_id"].iloc[0]
INSPECT_METHOD = "original"  # <- change to inspect other methods

row = df_results[(df_results["query_id"] == INSPECT_QUERY_ID) & (df_results["method"] == INSPECT_METHOD)].iloc[0]

print(f"Query ID:          {row['query_id']}")
print(f"Query:             {row['query']}")
print(f"Method:            {row['method']}")
print(f"Target position:   [{row['target_new_position']}] (0-based) out of {row['num_docs']} docs")
print(f"Citation order:    {row['citation_order']}")
print(f"Target cited at:   position {row['citation_order'].index(row['target_new_position']) + 1 if row['target_new_position'] in row['citation_order'] else 'NOT CITED'} in citation order")
print(f"\n--- LLM Response ---")
print(row["llm_response"][:800])

## Summary

In [ ]:
df_results = pd.read_parquet(output_path)

total_expected = len(all_query_ids) * len(all_methods)
print(f"Total expected:  {total_expected}")
print(f"Total done:      {len(df_results)}")
print(f"Remaining:       {total_expected - len(df_results)}")
print()

# Citation rate per method (was target cited at all?)
def was_target_cited(row):
    return 1 if row["boost_product_index"] in row["citation_order"] else 0

df_results["target_cited"] = df_results.apply(was_target_cited, axis=1)

citation_rates = df_results.groupby("method")["target_cited"].mean().sort_values(ascending=False)
print("Citation rate per method (% queries where target was cited):")
for method, rate in citation_rates.items():
    bar = "█" * int(rate * 20) + "░" * (20 - int(rate * 20))
    print(f"  {method:25} [{bar}] {rate:.1%}")